<a href="https://colab.research.google.com/github/KeshavGoyal-dot/UCS420-Cognitive-Computing/blob/main/UCS420_Assignment_4_FAQ_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCS420 – Cognitive Computing
## Assignment 4: A Cognitive FAQ System Using Pandas


### Q1. Build the Personalized Knowledge Base



In [1]:
import pandas as pd

roll_number = input("Enter your roll number: ").strip()

last_two = roll_number[-2:]
categories = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized_entries = []

for digit in last_two:
    d = int(digit)
    category = categories[d % 3]

    if category == "billing":
        entry = {
            "question": "how can i check my payment status",
            "answer": "You can check your payment status from the billing section.",
            "keywords": "payment status billing transaction",
            "category": category
        }
    elif category == "account":
        entry = {
            "question": "how do i update my registered mobile number",
            "answer": "Open Account Settings and update your registered mobile number.",
            "keywords": "mobile number update account",
            "category": category
        }
    else:
        entry = {
            "question": "where can i get general support",
            "answer": "You can contact the support desk for general help.",
            "keywords": "support help general contact",
            "category": category
        }

    personalized_entries.append(entry)

faq_data = fixed_entries + personalized_entries
df = pd.DataFrame(faq_data)

print("Final FAQ DataFrame:")
display(df)


Enter your roll number: 1024170318
Final FAQ DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Open Account Settings and update your register...,mobile number update account,account
5,where can i get general support,You can contact the support desk for general h...,support help general contact,general


### Q2. Generate and Score a Hypothesis



In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        score = sum(word in text for word in query_words)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return pd.DataFrame(results)


query = input("Enter a query: ")
matches = score_query(query, df)

if len(matches) == 0:
    print("No matching FAQ found.")
else:
    display(matches)


Enter a query: how are you?


,question,answer,category,score
0,how to reset password,Go to Settings > Reset Password.,account,1
1,what are your working hours,We are open 9 AM to 5 PM.,general,1
2,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1
3,how do i update my registered mobile number,Open Account Settings and update your register...,account,1


### Q3. Find FAQs from the Same Category




In [3]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][
        ["question", "answer", "keywords", "category"]
    ]


selected_category = personalized_entries[0]["category"]

print("Category:", selected_category)
display(same_category(selected_category, df))


Category: account


,question,answer,keywords,category
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
4,how do i update my registered mobile number,Open Account Settings and update your register...,mobile number update account,account


### Q4. Add a New Keyword and Save the Data




In [4]:
print("Choose an FAQ entry:")
for i, question in enumerate(df["question"]):
    print(i, "-", question)

entry_no = int(input("Enter the entry number: "))
new_keyword = input("Enter a new keyword: ").strip()

if new_keyword:
    df.loc[entry_no, "keywords"] = df.loc[entry_no, "keywords"] + " " + new_keyword

file_name = f"{roll_number}_faq_data.csv"
df.to_csv(file_name, index=False)

print("Updated entry:")
display(df.loc[[entry_no]])

print("Data saved as:", file_name)


Choose an FAQ entry:
0 - what is the annual fee
1 - how to reset password
2 - what are your working hours
3 - how can i pay the fee
4 - how do i update my registered mobile number
5 - where can i get general support
Enter the entry number: 2
Enter a new keyword: 5
Updated entry:


,question,answer,keywords,category
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time 5,general


Data saved as: 1024170318_faq_data.csv


### Q5. Number of FAQ Entries in Each Category


In [5]:
category_count = df.groupby("category").size()

print("FAQ entries per category:")
display(category_count)


FAQ entries per category:


,0
category,
account,2
billing,2
general,2


### Q6. Handle Ties in the Highest Score



In [6]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        score = sum(word in text for word in query_words)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if not results:
        print("No matching FAQ found.")
        return pd.DataFrame()

    results_df = pd.DataFrame(results)
    highest_score = results_df["score"].max()
    top_matches = results_df[results_df["score"] == highest_score]

    print("Highest score:", highest_score)
    if len(top_matches) > 1:
        print("There is a tie. All highest-scoring entries are shown:")
    else:
        print("There is one highest-scoring entry:")

    display(top_matches)
    return top_matches


#### Tie demonstration



In [7]:
tie_query = "fee"
print("Query:", tie_query)
tie_result = score_query_with_ties(tie_query, df)


Query: fee
Highest score: 1
There is a tie. All highest-scoring entries are shown:


,question,answer,category,score
0,what is the annual fee,The annual fee is Rs 500.,billing,1
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1


#### Non-tie demonstration


In [8]:
non_tie_query = "reset password"
print("Query:", non_tie_query)
non_tie_result = score_query_with_ties(non_tie_query, df)


Query: reset password
Highest score: 2
There is one highest-scoring entry:


,question,answer,category,score
0,how to reset password,Go to Settings > Reset Password.,account,2
